In [ ]:
import os
import math
import numpy as np
import xarray as xr
import scipy.ndimage
from pathlib import Path
import netCDF4 as nc
import cc3d
import gc
import metpy
from metpy.calc import density
from metpy.units import units
import matplotlib.pyplot as plt

negative_w_threshold = -0.25

In [ ]:
ds_ql = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/ql.nc", decode_times=False,chunks={'time': 1}) #liquid water mixing ratio
ds_w = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/w.nc", decode_times=False,chunks={'time': 1}).rename({'zh':'z'}).interp(z=ds_ql.z)

In [ ]:
#Making the shell
num_times = int(ds_ql.time.size)
nz, ny, nx = ds_ql.ql.shape[1:]
coords = ds_ql.coords

# Get the physical heights (z) as a 1D numpy array to map indices to meters
z_coordinates = ds_ql.z.values

#-Creating array for expansion
expansion = np.zeros((3,3,3), dtype=bool)
expansion[1, 1, :] = True  # X axis
expansion[1, :, 1] = True  # Y axis
expansion[:, 1, 1] = True  # Z axis

grid_shape = ds_ql.ql.shape

outline_mask = np.zeros(grid_shape, dtype=bool)
shell_w = np.zeros(grid_shape, dtype=np.float32)
shell_depth = np.full(grid_shape, np.nan, dtype=np.float32)

for t in range(num_times):
    print(f"\n--- Processing Timestep {t}/{num_times - 1} ---")
    ql_raw = (ds_ql.ql.isel(time=t).fillna(0) > 0).values.astype(bool)
    w_slice = (ds_w.w.isel(time=t).fillna(0) < negative_w_threshold).values.astype(bool)

    #Step 1 - Getting the shell
    # Dilated ql serving working space for shell expansion and intersection detection
    padded_ql = np.pad(ql_raw, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0) #padded z with 0's to prevent 
    padded_ql = np.pad(padded_ql, ((0, 0), (1, 1), (1, 1)), mode='wrap')
    padded_current = scipy.ndimage.binary_dilation(padded_ql, structure=expansion)
    current = padded_current[1:-1, 1:-1, 1:-1]

    padded_w = np.pad(w_slice, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)
    padded_labels = cc3d.connected_components(padded_w, connectivity=6, periodic_boundary=True) #labels every region in w
    labels = padded_labels[1:-1, :, :]
    
    num_features = np.max(labels)
    if num_features == 0:
        print(f"No objects found in timestep {t}. Skipping calculations.")
        continue

    #-get w regions that intersect with ql
    matching_labels = set(labels[current])
    matching_labels.discard(0)

    if not matching_labels:
        continue

    #-select w regions that connect to ql
    converged_mask = np.isin(labels, list(matching_labels))

    #-create outline and apply it to the mask
    outline_mask[t,:,:,:] = converged_mask & ~ql_raw
    local_outline_mask = outline_mask[t,:,:,:].astype(np.uint8)

    w_slice_physical = ds_w.w.isel(time=t).values
    shell_w[t,:,:,:] = np.where(outline_mask[t,:,:,:], w_slice_physical, np.nan)
    print(f"Shell constructed in timestep {t}.")

    #NEW - Shell Depth
    valid_shell_labels = np.where(outline_mask[t, :, :, :], labels, 0)

    slices = scipy.ndimage.find_objects(valid_shell_labels)

    for obj_id in matching_labels:
        obj_slice = slices[obj_id - 1] if obj_id - 1 < len(slices) else None

        if obj_slice is not None:
            z_slice = obj_slice[0]
            min_z_phys = z_coordinates[z_slice.start]
            max_z_phys = z_coordinates[z_slice.stop - 1]
            obj_depth = max_z_phys - min_z_phys
            full_4d_slice = (t, *obj_slice)
            box_mask = valid_shell_labels[obj_slice] == obj_id
            shell_depth[full_4d_slice][box_mask] = obj_depth

    print(f"Shell depths constructed in timestep {t}.")

In [ ]:
ds_export = xr.Dataset(
    data_vars={
        "shell_depth": (["time", "z", "y", "x"], shell_depth),
    },
    coords=ds_ql.coords
)

ds_export["shell_depth"].attrs["units"] = "meters"
ds_export["shell_depth"].attrs["long_name"] = "Total vertical thickness of individual shell"

output_directory = "/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area"
output_filename = os.path.join(output_directory, "shell_depth.nc")

os.makedirs(output_directory, exist_ok=True)

ds_export.to_netcdf(output_filename)

print(f"Successfully exported dataset to {output_filename}")